# BuzzSpot Model B — YOLO26m 700×700 sliced fine-tuning

This notebook trains **Model B**, the slice-specialized YOLO26m model.

Separation:

- **Model A** already exists: `yolo26m_32ep_rare_os_cm5_best.pt`
  - trained on full-frame images
  - used for full-frame inference
- **Model B** starts from Model A weights, then fine-tunes only on 700×700 SAHI-style slices
  - used for sliced inference
  - this notebook can create a **Model B-only** submission zip
- The **A+B ensemble/merge** belongs in a separate notebook after Model B finishes.

This notebook does **not** train on development/test data. Test images are only used for final inference/zip creation.

Fixes in this version:

- **Training empty-slice cap**: all positive slices are kept/oversampled, but empty/background slices are randomly capped to `EMPTY_TO_POS_RATIO × effective_positive_entries` in the training list. Empty validation slices are **not** capped.
- **Low confidence for AP measurement**: validation / COCOeval use `EVAL_CONF = 0.001`; test zip generation uses separate `SUBMIT_CONF = 0.001`.



## 1. Install


In [ ]:
!pip install -q ultralytics pycocotools tqdm

## 2. Mount Drive and locate dataset files


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import glob
from pathlib import Path

zip_hits = glob.glob("/content/drive/MyDrive/**/BuzzSet_challenge.zip", recursive=True)
assert zip_hits, "BuzzSet_challenge.zip not found under MyDrive"
ZIP_PATH = zip_hits[0]

tar_hits = glob.glob("/content/drive/MyDrive/**/19557529600-BuzzSet_challenge_testphase.tar", recursive=True)
if not tar_hits:
    tar_hits = glob.glob("/content/drive/MyDrive/**/*BuzzSet_challenge_testphase*.tar*", recursive=True)
assert tar_hits, "test-phase tar file not found under MyDrive"
TAR_PATH = tar_hits[0]

print("old train/valid zip:", ZIP_PATH)
print("new test-phase tar:", TAR_PATH)


## 3. Config


In [ ]:
from pathlib import Path
import shutil

ENABLE_TRAINING = True
REBUILD_SLICED_DATASET = True

MODEL_A_TAG = "yolo26m_32ep_rare_os_cm5"
MODEL_TAG = "yolo26m_sliced700_from_fullframe_12ep"
RUN_NAME = "buzz_yolo26m_sliced700_from_fullframe_12ep"

SLICE_SIZE = 700
OVERLAP = 0.20
STRIDE = int(SLICE_SIZE * (1.0 - OVERLAP))  # 560

MIN_VISIBLE_FRAC = 0.20
MIN_BOX_SIDE = 2.0

IMGSZ = 704
EPOCHS = 12
PATIENCE = 5
SAVE_PERIOD = 3
BATCH = 8  # if this OOMs on L4, use 4 or 6
SEED = 0

MOSAIC = 0.5
CLOSE_MOSAIC = 3

# Model B-only sliced inference settings.
# Use low confidence for AP/COCOeval so the PR curve is not truncated.
# Submission uses a separate variable so it can be changed without corrupting validation comparisons.
EVAL_CONF = 0.001
SUBMIT_CONF = 0.001
IOU = 0.45
MAX_DET_PER_SLICE = 100
AUGMENT = False

# Merge duplicate predictions from overlapping Model B slices.
# This is NOT Model A+B merging.
MERGE_METHOD = "nmm_ios"
MERGE_IOS = 0.50
MAX_DET_FINAL = 100
INFER_BATCH_SLICES = 8

LOCAL_DATA_DIR = Path("/content/buzz_modelB_sliced700")
SOURCE_DIR = LOCAL_DATA_DIR / "source_full"
SLICED_DIR = LOCAL_DATA_DIR / "sliced700"

LOCAL_WEIGHTS = Path("/content/modelB_best.pt")

PROJECT_DIR = Path("/content/drive/MyDrive/BuzzSpot")
WEIGHTS_DIR = PROJECT_DIR / "weights"
DRIVE_RUNS_DIR = PROJECT_DIR / "runs_sliced700"
SUBMISSIONS_DIR = PROJECT_DIR / "submissions"

WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_RUNS_DIR.mkdir(parents=True, exist_ok=True)
SUBMISSIONS_DIR.mkdir(parents=True, exist_ok=True)

BASE_FULLFRAME_WEIGHTS = WEIGHTS_DIR / f"{MODEL_A_TAG}_best.pt"
DRIVE_BEST_WEIGHTS = WEIGHTS_DIR / f"{MODEL_TAG}_best.pt"
DRIVE_LAST_WEIGHTS = WEIGHTS_DIR / f"{MODEL_TAG}_last.pt"
DRIVE_RESULTS_CSV = WEIGHTS_DIR / f"{MODEL_TAG}_results.csv"

EVAL_CONF_TAG = f"evalconf{int(EVAL_CONF * 1000):03d}"
SUBMIT_CONF_TAG = f"conf{int(SUBMIT_CONF * 1000):03d}"
LOCAL_PRED_PATH = Path("/content/predictions.json")
LOCAL_ZIP_OUT = Path(f"/content/submission_{MODEL_TAG}_modelB_only_{SUBMIT_CONF_TAG}.zip")
DRIVE_PRED_PATH = SUBMISSIONS_DIR / f"predictions_{MODEL_TAG}_modelB_only_{SUBMIT_CONF_TAG}.json"
DRIVE_ZIP_OUT = SUBMISSIONS_DIR / f"submission_{MODEL_TAG}_modelB_only_{SUBMIT_CONF_TAG}.zip"

CLASS_NAMES = ["bee", "bumblebee", "hoverfly", "moth"]

# Empty slices are useful hard background, but uncapped empty/background slices can dominate training.
# This cap applies only to the training TXT list; it does not delete files and it does not cap validation.
CAP_EMPTY_SLICES = True
EMPTY_TO_POS_RATIO = 3.0

CLASS_MULTIPLIER = {
    0: 1,  # bee
    1: 4,  # bumblebee
    2: 2,  # hoverfly
    3: 5,  # moth
}


def restore_best_weights_to_local():
    candidates = [
        DRIVE_BEST_WEIGHTS,
        DRIVE_RUNS_DIR / RUN_NAME / "weights" / "best.pt",
    ]

    for p in candidates:
        if p.exists():
            shutil.copy(p, LOCAL_WEIGHTS)
            print("Restored Model B best.pt from Drive:", p)
            print("Local weights:", LOCAL_WEIGHTS)
            return LOCAL_WEIGHTS

    if LOCAL_WEIGHTS.exists():
        print("WARNING: Drive Model B backup not found, but local modelB_best.pt exists:", LOCAL_WEIGHTS)
        return LOCAL_WEIGHTS

    raise FileNotFoundError(
        "No saved Model B best.pt found. Set ENABLE_TRAINING=True and run training first."
    )


def backup_training_outputs_to_drive():
    run_dir = DRIVE_RUNS_DIR / RUN_NAME
    best_path = run_dir / "weights" / "best.pt"
    last_path = run_dir / "weights" / "last.pt"
    results_csv = run_dir / "results.csv"

    assert best_path.exists(), f"best.pt not found at {best_path}"

    shutil.copy(best_path, LOCAL_WEIGHTS)
    shutil.copy(best_path, DRIVE_BEST_WEIGHTS)
    print("Copied Model B best weights to local:", LOCAL_WEIGHTS)
    print("Backed up Model B best weights to Drive:", DRIVE_BEST_WEIGHTS)

    if last_path.exists():
        shutil.copy(last_path, DRIVE_LAST_WEIGHTS)
        print("Backed up Model B last weights to Drive:", DRIVE_LAST_WEIGHTS)

    if results_csv.exists():
        shutil.copy(results_csv, DRIVE_RESULTS_CSV)
        print("Backed up Model B results.csv to Drive:", DRIVE_RESULTS_CSV)


print("ENABLE_TRAINING:", ENABLE_TRAINING)
print("REBUILD_SLICED_DATASET:", REBUILD_SLICED_DATASET)
print("Model A base weights:", BASE_FULLFRAME_WEIGHTS)
print("Model B tag:", MODEL_TAG)
print("slice size:", SLICE_SIZE)
print("overlap:", OVERLAP)
print("stride:", STRIDE)
print("imgsz:", IMGSZ)
print("epochs:", EPOCHS)
print("batch:", BATCH)
print("seed:", SEED)
print("mosaic:", MOSAIC)
print("close_mosaic:", CLOSE_MOSAIC)
print("eval conf:", EVAL_CONF)
print("submit conf:", SUBMIT_CONF)
print("max det per slice:", MAX_DET_PER_SLICE)
print("cap empty slices:", CAP_EMPTY_SLICES)
print("empty-to-positive ratio:", EMPTY_TO_POS_RATIO)
print("Model B Drive best weights:", DRIVE_BEST_WEIGHTS)
print("Model B-only Drive zip output:", DRIVE_ZIP_OUT)

assert BASE_FULLFRAME_WEIGHTS.exists(), (
    f"Base full-frame Model A checkpoint missing: {BASE_FULLFRAME_WEIGHTS}\n"
    "Run/restore buzzspot_yolo26m_32ep.ipynb first."
)


## 4. Build source full images and complete 700×700 sliced YOLO dataset


In [ ]:
import json, zipfile, tarfile, shutil, yaml
from pathlib import Path
from collections import defaultdict, Counter
from PIL import Image
from tqdm.auto import tqdm

if REBUILD_SLICED_DATASET and LOCAL_DATA_DIR.exists():
    shutil.rmtree(LOCAL_DATA_DIR)

for d in [
    SOURCE_DIR / "images/train",
    SOURCE_DIR / "images/val",
    SOURCE_DIR / "images/test_testphase",
    LOCAL_DATA_DIR / "annotations",
    SLICED_DIR / "images/train",
    SLICED_DIR / "images/val",
    SLICED_DIR / "labels/train",
    SLICED_DIR / "labels/val",
]:
    d.mkdir(parents=True, exist_ok=True)

zf = zipfile.ZipFile(ZIP_PATH)
tf = tarfile.open(TAR_PATH, "r:*")

zip_names = set(zf.namelist())
tar_members = {m.name: m for m in tf.getmembers() if m.isfile()}
tar_names = set(tar_members.keys())


def find_zip(rel):
    rel = rel.lstrip("/")
    for cand in (rel, f"BuzzSet_challenge/{rel}"):
        if cand in zip_names:
            return cand
    suffix = "/" + rel
    hits = [n for n in zip_names if n.endswith(suffix)]
    return hits[0] if hits else None


def find_tar(rel):
    rel = rel.lstrip("/")
    for cand in (rel, f"BuzzSet_challenge/{rel}", f"BuzzSet_challenge_testphase/{rel}"):
        if cand in tar_names:
            return cand
    suffix = "/" + rel
    hits = [n for n in tar_names if n.endswith(suffix)]
    return hits[0] if hits else None


def load_tar_json(fname):
    p = find_tar(f"annotations/{fname}") or find_tar(fname)
    assert p is not None, f"{fname} not found in tar"
    with tf.extractfile(tar_members[p]) as f:
        obj = json.load(f)
    out = LOCAL_DATA_DIR / "annotations" / fname
    out.write_text(json.dumps(obj))
    print("loaded current annotation:", fname, "from", p)
    return obj


def flat_name(file_name):
    return file_name.replace("/", "__")


def annotations_by_image(coco):
    by_img = defaultdict(list)
    for ann in coco.get("annotations", []):
        by_img[int(ann["image_id"])].append(ann)
    return by_img


def extract_source_split(coco, zip_img_dir, out_split):
    """Extract only supervised/annotated keyframes for train/val.

    This deliberately matches the known-good Model A logic:
    - build by_img from annotations
    - iterate only image ids that appear in annotations
    - do NOT treat unannotated/context frames in coco["images"] as negative images

    Empty/background examples for Model B must come from 700x700 slices
    of annotated keyframes, not from unlabeled full context frames.
    """
    by_img = annotations_by_image(coco)
    images_by_id = {int(im["id"]): im for im in coco["images"]}

    # Critical: same selection rule as Model A.
    image_ids = sorted(by_img.keys())

    print(f"\n{out_split} COCO image records:", len(coco["images"]))
    print(f"{out_split} annotated image ids used:", len(image_ids))
    print(f"{out_split} skipped unannotated/context image records:", len(coco["images"]) - len(image_ids))

    missing_image_records = [iid for iid in image_ids if iid not in images_by_id]
    assert not missing_image_records, (
        f"{out_split}: {len(missing_image_records)} annotation image_ids not found in coco['images']; "
        f"first missing ids: {missing_image_records[:10]}"
    )

    records = []
    class_counts = Counter()
    box_count = 0

    for iid in tqdm(image_ids, desc=f"extract {out_split} annotated full images"):
        im = images_by_id[iid]

        src = find_zip(f"{zip_img_dir}/{im['file_name']}") or find_zip(im["file_name"])
        assert src is not None, f"missing image in old zip: {zip_img_dir}/{im['file_name']}"

        out_name = flat_name(im["file_name"])
        img_dst = SOURCE_DIR / "images" / out_split / out_name

        if REBUILD_SLICED_DATASET or not img_dst.exists():
            with zf.open(src) as s, open(img_dst, "wb") as d:
                shutil.copyfileobj(s, d)

        anns = by_img[iid]
        assert len(anns) > 0, f"{out_split}: internal bug, selected image id {iid} has no annotations"

        for ann in anns:
            class_counts[int(ann["category_id"])] += 1
        box_count += len(anns)

        records.append({
            "id": iid,
            "file_name": im["file_name"],
            "flat_name": out_name,
            "path": str(img_dst),
            "width": int(im["width"]),
            "height": int(im["height"]),
            "annotations": anns,
        })

    print(out_split, "images:", len(records), "boxes:", box_count, "class counts:", dict(class_counts))
    print(out_split, "named counts:", {CLASS_NAMES[k-1]: class_counts[k] for k in range(1, 5)})

    expected_images = {"train": 5275, "val": 932}.get(out_split)
    expected_boxes = {"train": 10884, "val": 1116}.get(out_split)

    if expected_images is not None:
        assert len(records) == expected_images, (
            f"{out_split}: expected {expected_images} annotated images, got {len(records)}. "
            "Do not train until this matches Model A."
        )

    if expected_boxes is not None:
        assert box_count == expected_boxes, (
            f"{out_split}: expected {expected_boxes} boxes, got {box_count}. "
            "Do not train until this matches Model A/current annotations."
        )

    return records


def extract_test_keyframes(test_coco):
    keyframe_images = [
        im for im in test_coco["images"]
        if im.get("is_keyframe") in [True, 1, "true", "True"]
    ]

    records = []
    flat_to_id = {}

    for im in tqdm(keyframe_images, desc="extract test_testphase keyframes"):
        src = find_tar(f"test_testphase/{im['file_name']}") or find_tar(im["file_name"])
        assert src is not None, f"missing test image in tar: test_testphase/{im['file_name']}"

        out_name = flat_name(im["file_name"])
        dst = SOURCE_DIR / "images" / "test_testphase" / out_name

        if REBUILD_SLICED_DATASET or not dst.exists():
            with tf.extractfile(tar_members[src]) as s, open(dst, "wb") as d:
                shutil.copyfileobj(s, d)

        flat_to_id[out_name] = int(im["id"])
        records.append({
            "id": int(im["id"]),
            "file_name": im["file_name"],
            "flat_name": out_name,
            "path": str(dst),
            "width": int(im.get("width", 1920)),
            "height": int(im.get("height", 1080)),
        })

    print("test_testphase keyframes:", len(records))
    return records, flat_to_id


def slice_starts(length, size=SLICE_SIZE, stride=STRIDE):
    if length <= size:
        return [0]
    starts = list(range(0, length - size + 1, stride))
    last = length - size
    if starts[-1] != last:
        starts.append(last)
    return sorted(set(starts))


def clip_ann_to_slice(ann, x0, y0, slice_size=SLICE_SIZE):
    x, y, w, h = map(float, ann["bbox"])
    x1, y1, x2, y2 = x, y, x + w, y + h

    sx1, sy1 = x0, y0
    sx2, sy2 = x0 + slice_size, y0 + slice_size

    ix1 = max(x1, sx1)
    iy1 = max(y1, sy1)
    ix2 = min(x2, sx2)
    iy2 = min(y2, sy2)

    iw = ix2 - ix1
    ih = iy2 - iy1

    if iw < MIN_BOX_SIDE or ih < MIN_BOX_SIDE:
        return None

    orig_area = max(1e-9, w * h)
    inter_area = iw * ih
    visible_frac = inter_area / orig_area

    if visible_frac < MIN_VISIBLE_FRAC:
        return None

    rx1 = ix1 - x0
    ry1 = iy1 - y0
    rx2 = ix2 - x0
    ry2 = iy2 - y0

    bw = rx2 - rx1
    bh = ry2 - ry1
    xc = (rx1 + bw / 2.0) / slice_size
    yc = (ry1 + bh / 2.0) / slice_size
    ww = bw / slice_size
    hh = bh / slice_size

    cls = int(ann["category_id"]) - 1

    if cls < 0 or cls >= len(CLASS_NAMES):
        return None

    return cls, xc, yc, ww, hh


def make_sliced_split(records, split):
    img_out_dir = SLICED_DIR / "images" / split
    lbl_out_dir = SLICED_DIR / "labels" / split
    img_out_dir.mkdir(parents=True, exist_ok=True)
    lbl_out_dir.mkdir(parents=True, exist_ok=True)

    slice_paths = []
    slice_class_counts = Counter()
    slice_image_class_counts = Counter()
    empty_slices = 0
    positive_slices = 0
    total_boxes = 0

    for rec in tqdm(records, desc=f"make {split} 700x700 slices"):
        img_path = Path(rec["path"])
        img = Image.open(img_path).convert("RGB")
        W, H = img.size

        xs = slice_starts(W)
        ys = slice_starts(H)

        base_stem = Path(rec["flat_name"]).stem

        for y0 in ys:
            for x0 in xs:
                slice_name = f"{base_stem}__x{x0}_y{y0}.jpg"
                slice_img_path = img_out_dir / slice_name
                slice_lbl_path = lbl_out_dir / f"{Path(slice_name).stem}.txt"

                if REBUILD_SLICED_DATASET or not slice_img_path.exists():
                    crop = img.crop((x0, y0, x0 + SLICE_SIZE, y0 + SLICE_SIZE))
                    crop.save(slice_img_path, quality=95)

                lines = []
                classes_in_slice = set()

                for ann in rec["annotations"]:
                    clipped = clip_ann_to_slice(ann, x0, y0)
                    if clipped is None:
                        continue

                    cls, xc, yc, ww, hh = clipped
                    lines.append(f"{cls} {xc:.6f} {yc:.6f} {ww:.6f} {hh:.6f}")
                    classes_in_slice.add(cls)
                    slice_class_counts[cls] += 1
                    total_boxes += 1

                slice_lbl_path.write_text("\n".join(lines))

                if classes_in_slice:
                    positive_slices += 1
                    for c in classes_in_slice:
                        slice_image_class_counts[c] += 1
                else:
                    empty_slices += 1

                slice_paths.append(slice_img_path)

        img.close()

    print(f"\\n{split} sliced images:", len(slice_paths))
    print(f"{split} positive slices:", positive_slices)
    print(f"{split} empty slices:", empty_slices)
    print(f"{split} boxes after clipping:", total_boxes)
    print(f"{split} instance counts:", {CLASS_NAMES[k]: slice_class_counts[k] for k in range(4)})
    print(f"{split} slice-image counts:", {CLASS_NAMES[k]: slice_image_class_counts[k] for k in range(4)})

    return slice_paths


def label_classes(label_path):
    cls_set = set()
    if not Path(label_path).exists():
        return cls_set
    for line in Path(label_path).read_text().splitlines():
        parts = line.strip().split()
        if len(parts) >= 5:
            cls_set.add(int(float(parts[0])))
    return cls_set


def label_instance_counts(label_path):
    cnt = Counter()
    if not Path(label_path).exists():
        return cnt
    for line in Path(label_path).read_text().splitlines():
        parts = line.strip().split()
        if len(parts) >= 5:
            cnt[int(float(parts[0]))] += 1
    return cnt


def build_oversampled_train_txt(train_slice_paths):
    """Build the actual YOLO training list.

    Important: all slice image/label files remain on disk. This function only controls which
    image paths YOLO sees during training.

    Policy:
    - keep every positive slice
    - oversample positive slices using CLASS_MULTIPLIER
    - randomly sample empty/background slices up to EMPTY_TO_POS_RATIO × effective positive entries
    - do not cap validation; this function is train-only
    """
    import random

    rng = random.Random(SEED)

    positive_entries = []
    empty_paths = []

    unique_count = 0
    image_type_counts = Counter()
    original_instance_counts = Counter()
    effective_instance_counts = Counter()
    positive_unique_by_class = Counter()
    positive_effective_by_class = Counter()

    for img_path in train_slice_paths:
        img_path = Path(img_path)
        lbl_path = SLICED_DIR / "labels" / "train" / (img_path.stem + ".txt")
        cls_set = label_classes(lbl_path)
        inst_counts = label_instance_counts(lbl_path)
        unique_count += 1

        if cls_set:
            mult = max(CLASS_MULTIPLIER.get(c, 1) for c in cls_set)
            image_type_counts["positive_unique"] += 1
            for c in cls_set:
                positive_unique_by_class[c] += 1
                positive_effective_by_class[c] += mult

            original_instance_counts.update(inst_counts)

            for _ in range(mult):
                positive_entries.append(str(img_path.resolve()))
                effective_instance_counts.update(inst_counts)
        else:
            image_type_counts["empty_unique"] += 1
            empty_paths.append(str(img_path.resolve()))

    effective_positive_count = len(positive_entries)

    if CAP_EMPTY_SLICES:
        max_empty = int(effective_positive_count * EMPTY_TO_POS_RATIO)
        if len(empty_paths) > max_empty:
            selected_empty_paths = rng.sample(empty_paths, max_empty)
        else:
            selected_empty_paths = list(empty_paths)
    else:
        max_empty = len(empty_paths)
        selected_empty_paths = list(empty_paths)

    lines = positive_entries + selected_empty_paths
    rng.shuffle(lines)

    train_txt = SLICED_DIR / "train_sliced700_oversampled_empty_capped.txt"
    train_txt.write_text("\n".join(lines) + "\n")

    print("\ntrain unique slices:", unique_count)
    print("positive unique slices:", image_type_counts["positive_unique"])
    print("empty unique slices:", image_type_counts["empty_unique"])
    print("effective positive entries after oversampling:", effective_positive_count)
    print("empty cap enabled:", CAP_EMPTY_SLICES)
    print("EMPTY_TO_POS_RATIO:", EMPTY_TO_POS_RATIO)
    print("max empty allowed:", max_empty)
    print("selected empty entries:", len(selected_empty_paths))
    print("final train lines:", len(lines))
    print("final empty:positive ratio:", round(len(selected_empty_paths) / max(1, effective_positive_count), 3))
    print("positive unique slice counts by class:", {CLASS_NAMES[k]: positive_unique_by_class[k] for k in range(4)})
    print("positive effective slice entries by class:", {CLASS_NAMES[k]: positive_effective_by_class[k] for k in range(4)})
    print("original instance counts:", {CLASS_NAMES[k]: original_instance_counts[k] for k in range(4)})
    print("effective instance counts:", {CLASS_NAMES[k]: effective_instance_counts[k] for k in range(4)})
    print("train txt:", train_txt)

    if len(selected_empty_paths) == 0:
        print("WARNING: no empty/background slices selected. That is probably too aggressive.")
    if effective_positive_count == 0:
        raise RuntimeError("No positive training slices found. Check annotation remapping / slice generation.")

    return train_txt


train_coco = load_tar_json("train.json")
valid_coco = load_tar_json("valid.json")
test_coco = load_tar_json("test_testphase.json")

train_records = extract_source_split(train_coco, "train", "train")
valid_records = extract_source_split(valid_coco, "valid", "val")
test_records, flat_to_id = extract_test_keyframes(test_coco)

train_slice_paths = make_sliced_split(train_records, "train")
val_slice_paths = make_sliced_split(valid_records, "val")

TRAIN_TXT = build_oversampled_train_txt(train_slice_paths)

DATA_YAML = SLICED_DIR / "buzz_sliced700.yaml"
DATA_YAML.write_text(
    "\\n".join([
        f"path: {SLICED_DIR}",
        f"train: {TRAIN_TXT}",
        "val: images/val",
        "names:",
        "  0: bee",
        "  1: bumblebee",
        "  2: hoverfly",
        "  3: moth",
        ""
    ])
)

print("\\nDATA_YAML:")
print(DATA_YAML.read_text())


In [ ]:
# Fix malformed YAML: rewrite with real newlines, not literal "\n"

DATA_YAML = SLICED_DIR / "buzz_sliced700.yaml"

yaml_text = "\n".join([
    f"path: {SLICED_DIR}",
    f"train: {TRAIN_TXT}",
    "val: images/val",
    "names:",
    "  0: bee",
    "  1: bumblebee",
    "  2: hoverfly",
    "  3: moth",
    ""
])

DATA_YAML.write_text(yaml_text)

print("RAW REPR:")
print(repr(DATA_YAML.read_text()))

print("\nNORMAL PRINT:")
print(DATA_YAML.read_text())

# Hard fail if literal backslash-n remains
assert "\\n" not in DATA_YAML.read_text(), "YAML still contains literal backslash-n characters"
assert "\ntrain:" in DATA_YAML.read_text(), "YAML does not contain real newline before train"
assert DATA_YAML.exists()

## 5. Train Model B from Model A checkpoint on sliced dataset


In [ ]:
from ultralytics import YOLO
import torch, gc

if ENABLE_TRAINING:
    print("ENABLE_TRAINING=True -> training Model B will run.")
    print("Starting from Model A checkpoint:", BASE_FULLFRAME_WEIGHTS)

    model = YOLO(str(BASE_FULLFRAME_WEIGHTS))

    train_kwargs = dict(
        data=str(DATA_YAML),
        epochs=EPOCHS,
        imgsz=IMGSZ,
        batch=BATCH,
        patience=PATIENCE,
        save_period=SAVE_PERIOD,
        project=str(DRIVE_RUNS_DIR),
        name=RUN_NAME,
        exist_ok=True,
        seed=SEED,
        deterministic=True,
        cache="disk",
        workers=4,
        device=0,
        optimizer="auto",
        mosaic=MOSAIC,
        close_mosaic=CLOSE_MOSAIC,
    )

    print("training args:", train_kwargs)
    model.train(**train_kwargs)

    backup_training_outputs_to_drive()

    gc.collect()
    torch.cuda.empty_cache()

else:
    print("ENABLE_TRAINING=False -> skipping Model B training.")
    restore_best_weights_to_local()


## 6. Validate Model B on sliced val dataset


In [ ]:
from ultralytics import YOLO
from pathlib import Path
import pandas as pd

restore_best_weights_to_local()

model = YOLO(str(LOCAL_WEIGHTS))
print("classes:", model.names)

print("\nSliced-val YOLO validation config:")
print("EVAL_CONF:", EVAL_CONF)
print("IOU:", IOU)
print("MAX_DET_PER_SLICE:", MAX_DET_PER_SLICE)
print("IMGSZ:", IMGSZ)

metrics = model.val(
    data=str(DATA_YAML),
    imgsz=IMGSZ,
    batch=BATCH,
    split="val",
    conf=EVAL_CONF,
    iou=IOU,
    max_det=MAX_DET_PER_SLICE,
    agnostic_nms=False,
    plots=False,
    verbose=False,
)

row = {
    "eval_conf": EVAL_CONF,
    "iou": IOU,
    "max_det_per_slice": MAX_DET_PER_SLICE,
    "map50_95_sliced_val": float(metrics.box.map),
    "map50_sliced_val": float(metrics.box.map50),
    "map75_sliced_val": float(metrics.box.map75),
}

for cls_idx, cls_name in model.names.items():
    row[f"ap_{cls_name}"] = float(metrics.box.maps[cls_idx])

df = pd.DataFrame([row])
display(df)


## 7. Model B-only sliced inference on original validation images + COCOeval


In [ ]:
import json, gc, torch
from pathlib import Path
from collections import Counter
import numpy as np
from PIL import Image
from tqdm.auto import tqdm
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
from ultralytics import YOLO

restore_best_weights_to_local()
model = YOLO(str(LOCAL_WEIGHTS))

VALID_JSON_PATH = LOCAL_DATA_DIR / "annotations" / "valid.json"
VAL_B_ONLY_PRED_PATH = LOCAL_DATA_DIR / f"val_predictions_{MODEL_TAG}_modelB_only_{EVAL_CONF_TAG}.json"


def xyxy_area(box):
    x1, y1, x2, y2 = box
    return max(0.0, x2 - x1) * max(0.0, y2 - y1)


def intersection_area(a, b):
    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b
    ix1 = max(ax1, bx1)
    iy1 = max(ay1, by1)
    ix2 = min(ax2, bx2)
    iy2 = min(ay2, by2)
    return max(0.0, ix2 - ix1) * max(0.0, iy2 - iy1)


def ios_xyxy(a, b):
    inter = intersection_area(a, b)
    denom = min(xyxy_area(a), xyxy_area(b))
    if denom <= 0:
        return 0.0
    return inter / denom


def weighted_merge_cluster(cluster):
    scores = np.array([p["score"] for p in cluster], dtype=np.float64)
    boxes = np.array([p["bbox_xyxy"] for p in cluster], dtype=np.float64)

    if scores.sum() <= 0:
        weights = np.ones_like(scores) / len(scores)
    else:
        weights = scores / scores.sum()

    merged_box = (boxes * weights[:, None]).sum(axis=0).tolist()
    merged_score = float(scores.max())

    p0 = cluster[0].copy()
    p0["bbox_xyxy"] = merged_box
    p0["score"] = merged_score
    return p0


def merge_predictions_greedy_nmm_ios(raw_preds, ios_thr=MERGE_IOS, max_det=MAX_DET_FINAL):
    final = []

    for category_id in sorted(set(p["category_id"] for p in raw_preds)):
        cls_preds = [p for p in raw_preds if p["category_id"] == category_id]
        cls_preds = sorted(cls_preds, key=lambda p: p["score"], reverse=True)

        while cls_preds:
            top = cls_preds[0]
            cluster = [top]
            rest = []

            for p in cls_preds[1:]:
                if ios_xyxy(top["bbox_xyxy"], p["bbox_xyxy"]) >= ios_thr:
                    cluster.append(p)
                else:
                    rest.append(p)

            final.append(weighted_merge_cluster(cluster))
            cls_preds = rest

    final = sorted(final, key=lambda p: p["score"], reverse=True)[:max_det]
    return final


def clip_xyxy(x1, y1, x2, y2, W, H):
    x1 = max(0.0, min(float(x1), W - 1))
    y1 = max(0.0, min(float(y1), H - 1))
    x2 = max(0.0, min(float(x2), W))
    y2 = max(0.0, min(float(y2), H))
    return x1, y1, x2, y2


def predict_image_model_b_sliced(img_path, image_id, W=None, H=None, infer_conf=EVAL_CONF):
    img_path = Path(img_path)
    img = Image.open(img_path).convert("RGB")

    if W is None or H is None:
        W, H = img.size

    raw_preds = []
    jobs = []
    batch_imgs = []

    xs = slice_starts(W)
    ys = slice_starts(H)

    def flush_batch(batch_imgs, jobs):
        batch_raw = []
        if not batch_imgs:
            return batch_raw

        results = model.predict(
            source=batch_imgs,
            imgsz=IMGSZ,
            conf=infer_conf,
            iou=IOU,
            max_det=MAX_DET_PER_SLICE,
            batch=len(batch_imgs),
            augment=AUGMENT,
            agnostic_nms=False,
            verbose=False,
        )

        for r, (sx, sy) in zip(results, jobs):
            if r.boxes is None or len(r.boxes) == 0:
                continue

            for b in r.boxes:
                x1, y1, x2, y2 = b.xyxy[0].tolist()
                x1, y1, x2, y2 = x1 + sx, y1 + sy, x2 + sx, y2 + sy
                x1, y1, x2, y2 = clip_xyxy(x1, y1, x2, y2, W, H)

                if x2 - x1 >= 1 and y2 - y1 >= 1:
                    batch_raw.append({
                        "image_id": int(image_id),
                        "category_id": int(b.cls[0]) + 1,
                        "bbox_xyxy": [x1, y1, x2, y2],
                        "score": float(b.conf[0]),
                    })

        return batch_raw

    for y0 in ys:
        for x0 in xs:
            crop = img.crop((x0, y0, x0 + SLICE_SIZE, y0 + SLICE_SIZE))

            # PIL gives RGB. Ultralytics NumPy prediction expects OpenCV-style BGR.
            crop_bgr = np.ascontiguousarray(np.array(crop)[:, :, ::-1])

            batch_imgs.append(crop_bgr)
            jobs.append((x0, y0))

            if len(batch_imgs) == INFER_BATCH_SLICES:
                raw_preds.extend(flush_batch(batch_imgs, jobs))
                batch_imgs = []
                jobs = []

    if batch_imgs:
        raw_preds.extend(flush_batch(batch_imgs, jobs))

    img.close()

    merged = merge_predictions_greedy_nmm_ios(raw_preds, ios_thr=MERGE_IOS, max_det=MAX_DET_FINAL)

    coco_preds = []
    for p in merged:
        x1, y1, x2, y2 = p["bbox_xyxy"]
        coco_preds.append({
            "image_id": int(p["image_id"]),
            "category_id": int(p["category_id"]),
            "bbox": [round(x1, 2), round(y1, 2), round(x2 - x1, 2), round(y2 - y1, 2)],
            "score": round(float(p["score"]), 5),
        })

    return coco_preds, len(raw_preds), len(merged)


print("\nModel B-only original-val sliced inference config:")
print("EVAL_CONF:", EVAL_CONF)
print("IOU:", IOU)
print("MAX_DET_PER_SLICE:", MAX_DET_PER_SLICE)
print("MERGE_IOS:", MERGE_IOS)
print("MAX_DET_FINAL:", MAX_DET_FINAL)

val_preds = []
raw_total = 0
merged_total = 0

for i, rec in enumerate(tqdm(valid_records, desc="Model B sliced inference on original val")):
    preds_i, raw_i, merged_i = predict_image_model_b_sliced(
        rec["path"],
        image_id=rec["id"],
        W=rec["width"],
        H=rec["height"],
        infer_conf=EVAL_CONF,
    )
    val_preds.extend(preds_i)
    raw_total += raw_i
    merged_total += merged_i

    if i % 100 == 0:
        print(i, "/", len(valid_records), "raw:", raw_total, "merged:", merged_total)
        gc.collect()
        torch.cuda.empty_cache()

VAL_B_ONLY_PRED_PATH.write_text(json.dumps(val_preds))
print("\\nModel B-only original-val raw preds before merge:", raw_total)
print("Model B-only original-val preds after merge:", len(val_preds))
print("preds/image after merge:", len(val_preds) / max(1, len(valid_records)))
print("class counts:", dict(Counter(int(p["category_id"]) for p in val_preds)))
print("val pred path:", VAL_B_ONLY_PRED_PATH)

coco_gt = COCO(str(VALID_JSON_PATH))
coco_gt.dataset.setdefault("info", {})
coco_gt.dataset.setdefault("licenses", [])

if len(val_preds) == 0:
    raise RuntimeError("No validation predictions produced; cannot run COCOeval.")

coco_dt = coco_gt.loadRes(str(VAL_B_ONLY_PRED_PATH))
coco_eval = COCOeval(coco_gt, coco_dt, "bbox")
coco_eval.evaluate()
coco_eval.accumulate()
coco_eval.summarize()

print("\\nCOCOeval stats:")
print("AP50-95:", coco_eval.stats[0])
print("AP50:", coco_eval.stats[1])
print("AP75:", coco_eval.stats[2])


In [ ]:
import json
from collections import Counter
p = json.loads(VAL_B_ONLY_PRED_PATH.read_text())
hv = [x["score"] for x in p if x["category_id"]==3]
import numpy as np
hv=np.array(hv)
print("hoverfly preds:", len(hv))
for t in [0.001,0.01,0.05,0.1,0.25,0.5]:
    print(f"conf>={t}: {(hv>=t).sum()}")

In [ ]:
# 7b. Confidence sweep on saved Model B-only original-val predictions
# Uses the existing Cell 7 prediction JSON. Does NOT rerun inference.

import json
from pathlib import Path
from collections import Counter
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

PRED_JSON = Path(f"/content/buzz_modelB_sliced700/val_predictions_{MODEL_TAG}_modelB_only_evalconf001.json")
GT_JSON = LOCAL_DATA_DIR / "annotations" / "valid.json"

print("PRED_JSON:", PRED_JSON)
print("exists:", PRED_JSON.exists())
assert PRED_JSON.exists(), "Run Cell 7 first so the prediction JSON exists."

all_preds = json.loads(PRED_JSON.read_text())
coco_gt = COCO(str(GT_JSON))
img_ids = sorted(coco_gt.getImgIds())

print("all preds:", len(all_preds))
print("preds/image:", len(all_preds) / max(1, len(img_ids)))

sweep_rows = []

for thr in [0.001, 0.002, 0.003, 0.005, 0.0075, 0.01, 0.015, 0.02, 0.03, 0.05, 0.075, 0.10, 0.15, 0.20]:
    filtered = [p for p in all_preds if float(p["score"]) >= thr]

    cls_counts = Counter(int(p["category_id"]) for p in filtered)

    if len(filtered) == 0:
        row = {
            "thr": thr,
            "preds": 0,
            "preds_per_image": 0.0,
            "AP50_95": 0.0,
            "AP50": 0.0,
            "AP75": 0.0,
            "bee": 0,
            "bumblebee": 0,
            "hoverfly": 0,
            "moth": 0,
        }
        sweep_rows.append(row)
        continue

    tmp_path = Path(f"/content/tmp_modelB_val_thr_{str(thr).replace('.', 'p')}.json")
    tmp_path.write_text(json.dumps(filtered))

    coco_dt = coco_gt.loadRes(str(tmp_path))
    ev = COCOeval(coco_gt, coco_dt, "bbox")
    ev.params.imgIds = img_ids
    ev.evaluate()
    ev.accumulate()
    ev.summarize()

    row = {
        "thr": thr,
        "preds": len(filtered),
        "preds_per_image": len(filtered) / max(1, len(img_ids)),
        "AP50_95": float(ev.stats[0]),
        "AP50": float(ev.stats[1]),
        "AP75": float(ev.stats[2]),
        "bee": cls_counts.get(1, 0),
        "bumblebee": cls_counts.get(2, 0),
        "hoverfly": cls_counts.get(3, 0),
        "moth": cls_counts.get(4, 0),
    }
    sweep_rows.append(row)

# Pretty print without needing pandas
print("\nCONF SWEEP SUMMARY")
print(
    f"{'thr':>8} {'preds':>8} {'pred/img':>9} "
    f"{'AP50-95':>9} {'AP50':>9} {'AP75':>9} "
    f"{'bee':>7} {'bumble':>7} {'hover':>7} {'moth':>7}"
)
for r in sweep_rows:
    print(
        f"{r['thr']:8.4f} {r['preds']:8d} {r['preds_per_image']:9.3f} "
        f"{r['AP50_95']:9.4f} {r['AP50']:9.4f} {r['AP75']:9.4f} "
        f"{r['bee']:7d} {r['bumblebee']:7d} {r['hoverfly']:7d} {r['moth']:7d}"
    )

best_ap = max(sweep_rows, key=lambda r: r["AP50_95"])
best_ap50 = max(sweep_rows, key=lambda r: r["AP50"])

print("\nBest by AP50-95:", best_ap)
print("Best by AP50:", best_ap50)

## 8. Model B-only sliced inference on test_testphase + zip


In [ ]:
import json, zipfile, gc, torch, shutil
from pathlib import Path
from ultralytics import YOLO
from tqdm.auto import tqdm
from collections import Counter

restore_best_weights_to_local()
model = YOLO(str(LOCAL_WEIGHTS))

gc.collect()
torch.cuda.empty_cache()

print("\nModel B-only test sliced inference config:")
print("SUBMIT_CONF:", SUBMIT_CONF)
print("IOU:", IOU)
print("MAX_DET_PER_SLICE:", MAX_DET_PER_SLICE)
print("MERGE_IOS:", MERGE_IOS)
print("MAX_DET_FINAL:", MAX_DET_FINAL)

test_preds = []
raw_total = 0
merged_total = 0

for i, rec in enumerate(tqdm(test_records, desc="Model B-only sliced inference on test")):
    preds_i, raw_i, merged_i = predict_image_model_b_sliced(
        rec["path"],
        image_id=rec["id"],
        W=rec["width"],
        H=rec["height"],
        infer_conf=SUBMIT_CONF,
    )

    test_preds.extend(preds_i)
    raw_total += raw_i
    merged_total += merged_i

    if i % 100 == 0:
        print(i, "/", len(test_records), "raw:", raw_total, "merged:", len(test_preds))
        gc.collect()
        torch.cuda.empty_cache()

with open(LOCAL_PRED_PATH, "w") as f:
    json.dump(test_preds, f)

with zipfile.ZipFile(LOCAL_ZIP_OUT, "w", zipfile.ZIP_DEFLATED) as z:
    z.write(LOCAL_PRED_PATH, arcname="predictions.json")

shutil.copy(LOCAL_PRED_PATH, DRIVE_PRED_PATH)
shutil.copy(LOCAL_ZIP_OUT, DRIVE_ZIP_OUT)

print("\\nraw test preds before merge:", raw_total)
print("detections after merge:", len(test_preds))
print("preds/image after merge:", len(test_preds) / max(1, len(test_records)))
print("class counts:", dict(Counter(int(p["category_id"]) for p in test_preds)))
print("local predictions:", LOCAL_PRED_PATH)
print("local zip:", LOCAL_ZIP_OUT)
print("Drive predictions:", DRIVE_PRED_PATH)
print("Drive zip:", DRIVE_ZIP_OUT)


## 9. Validate Model B-only submission zip before upload


In [ ]:
import json, zipfile
from pathlib import Path
from collections import Counter

p = json.load(open(LOCAL_PRED_PATH))
ids = {int(d["image_id"]) for d in p}
keyframe_ids = {int(rec["id"]) for rec in test_records}

print("detections:", len(p))
print("distinct predicted images:", len(ids))
print("total keyframe images:", len(keyframe_ids))
print("predictions outside keyframes:", len(ids - keyframe_ids))
print("keyframes with no detections:", len(keyframe_ids - ids))
print("categories:", sorted({int(d["category_id"]) for d in p}))
print("class counts:", dict(Counter(int(d["category_id"]) for d in p)))

print("degenerate:", sum(1 for d in p if d["bbox"][2] < 1 or d["bbox"][3] < 1))
print("out of bounds:", sum(
    1 for d in p
    if d["bbox"][0] < -0.5
    or d["bbox"][1] < -0.5
    or d["bbox"][0] + d["bbox"][2] > 1920.5
    or d["bbox"][1] + d["bbox"][3] > 1080.5
))

with zipfile.ZipFile(LOCAL_ZIP_OUT) as z:
    contents = z.namelist()
    print("zip contents:", contents)

assert len(ids - keyframe_ids) == 0, "Submission has predictions outside keyframes"
assert sorted({int(d["category_id"]) for d in p}) and set(int(d["category_id"]) for d in p).issubset({1,2,3,4}), "Bad category ids"
assert all(d["bbox"][2] >= 1 and d["bbox"][3] >= 1 for d in p), "Degenerate boxes exist"
assert contents == ["predictions.json"], "Zip must contain exactly predictions.json"

print("\nUpload this Model B-only zip if you want to test sliced model alone:")
print(DRIVE_ZIP_OUT)


## 10. Optional visual sanity check: GT vs Model B-only sliced predictions on original val


In [ ]:
# Run this only if you want visual inspection after validation.
# Green = GT. Red = Model B-only sliced predictions after slice-merge.

import random, gc, torch
from pathlib import Path
from PIL import Image, ImageDraw, ImageFont
import matplotlib.pyplot as plt

NUM_RANDOM_IMAGES = 50
RANDOM_SEED = 7
ZOOM_TO_BOXES = True
CROP_MARGIN = 180
MIN_CROP_SIZE = 500
LINE_WIDTH = 5
FONT_SIZE = 24

GT_COLOR = (0, 255, 0)
PRED_COLOR = (255, 0, 0)

try:
    FONT = ImageFont.truetype("DejaVuSans-Bold.ttf", FONT_SIZE)
except:
    FONT = None


def gt_boxes_from_record(rec):
    boxes = []
    for ann in rec["annotations"]:
        x, y, w, h = ann["bbox"]
        boxes.append({
            "category_id": int(ann["category_id"]),
            "bbox_xyxy": [x, y, x + w, y + h],
            "score": None,
        })
    return boxes


def coco_preds_to_xyxy(preds):
    out = []
    for p in preds:
        x, y, w, h = p["bbox"]
        out.append({
            "category_id": int(p["category_id"]),
            "bbox_xyxy": [x, y, x + w, y + h],
            "score": float(p["score"]),
        })
    return out


def get_crop_box(img, gt_boxes, pred_boxes):
    W, H = img.size
    all_boxes = gt_boxes + pred_boxes

    if not ZOOM_TO_BOXES or len(all_boxes) == 0:
        return (0, 0, W, H)

    xs1 = [b["bbox_xyxy"][0] for b in all_boxes]
    ys1 = [b["bbox_xyxy"][1] for b in all_boxes]
    xs2 = [b["bbox_xyxy"][2] for b in all_boxes]
    ys2 = [b["bbox_xyxy"][3] for b in all_boxes]

    x1 = max(0, int(min(xs1) - CROP_MARGIN))
    y1 = max(0, int(min(ys1) - CROP_MARGIN))
    x2 = min(W, int(max(xs2) + CROP_MARGIN))
    y2 = min(H, int(max(ys2) + CROP_MARGIN))

    if x2 - x1 < MIN_CROP_SIZE:
        extra = (MIN_CROP_SIZE - (x2 - x1)) // 2
        x1 = max(0, x1 - extra)
        x2 = min(W, x2 + extra)

    if y2 - y1 < MIN_CROP_SIZE:
        extra = (MIN_CROP_SIZE - (y2 - y1)) // 2
        y1 = max(0, y1 - extra)
        y2 = min(H, y2 + extra)

    return (x1, y1, x2, y2)


def draw_boxes(img, boxes, crop_box, color, is_prediction=False):
    out = img.crop(crop_box).convert("RGB")
    draw = ImageDraw.Draw(out)

    ox, oy, _, _ = crop_box

    for b in boxes:
        x1, y1, x2, y2 = b["bbox_xyxy"]
        x1 -= ox
        x2 -= ox
        y1 -= oy
        y2 -= oy

        cls_id = int(b["category_id"])
        cls_name = CLASS_NAMES[cls_id - 1]

        if is_prediction:
            label = f"{cls_name} {b['score']:.3f}"
        else:
            label = f"GT {cls_name}"

        draw.rectangle([x1, y1, x2, y2], outline=color, width=LINE_WIDTH)

        text_x = max(0, int(x1))
        text_y = max(0, int(y1) - FONT_SIZE - 8)

        if FONT is not None:
            tb = draw.textbbox((text_x, text_y), label, font=FONT)
            draw.rectangle([tb[0] - 4, tb[1] - 4, tb[2] + 4, tb[3] + 4], fill=color)
            draw.text((text_x, text_y), label, fill=(0, 0, 0), font=FONT)
        else:
            draw.rectangle([text_x, text_y, text_x + 220, text_y + 28], fill=color)
            draw.text((text_x + 3, text_y + 3), label, fill=(0, 0, 0))

    return out


rng = random.Random(RANDOM_SEED)
sample_records = rng.sample(valid_records, k=min(NUM_RANDOM_IMAGES, len(valid_records)))

print("Showing", len(sample_records), "random original val images")
print("Mode: Model B-only sliced inference")
print("EVAL_CONF:", EVAL_CONF, "IOU:", IOU, "MERGE_IOS:", MERGE_IOS)

for idx, rec in enumerate(sample_records, start=1):
    img = Image.open(rec["path"]).convert("RGB")
    gt_boxes = gt_boxes_from_record(rec)

    preds_i, raw_i, merged_i = predict_image_model_b_sliced(
        rec["path"],
        image_id=rec["id"],
        W=rec["width"],
        H=rec["height"],
        infer_conf=EVAL_CONF,
    )
    pred_boxes = coco_preds_to_xyxy(preds_i)

    crop_box = get_crop_box(img, gt_boxes, pred_boxes)

    gt_view = draw_boxes(img, gt_boxes, crop_box, GT_COLOR, is_prediction=False)
    pred_view = draw_boxes(img, pred_boxes, crop_box, PRED_COLOR, is_prediction=True)

    print(
        f"\\n[{idx}/{len(sample_records)}] {Path(rec['path']).name} | "
        f"GT={len(gt_boxes)} | raw slice preds={raw_i} | merged preds={len(pred_boxes)}"
    )

    fig, axes = plt.subplots(1, 2, figsize=(24, 9), dpi=120)
    axes[0].imshow(gt_view)
    axes[0].set_title(f"Ground Truth | {Path(rec['path']).name}", fontsize=16)
    axes[0].axis("off")

    axes[1].imshow(pred_view)
    axes[1].set_title("Prediction | Model B-only sliced", fontsize=16)
    axes[1].axis("off")

    plt.tight_layout()
    plt.show()

    img.close()

    if idx % 10 == 0:
        gc.collect()
        torch.cuda.empty_cache()
